In [1]:
from typing import List, Dict, Tuple, Any
import os, json

In [2]:
def calculate_kg_similarity(kg1: List[Dict], kg2: List[Dict]) -> float:
    # If both are empty, they are perfectly identical
    if not kg1 and not kg2:
        return 1.0
    
    # If only one is empty, there is zero overlap
    if not kg1 or not kg2:
        return 0.0

    def extract_core_relations(kg: List[Dict]) -> set:
        """Converts dict relations into lowercase, hashable tuples for exact set matching."""
        return set(
            (
                rel.get('head', '').strip().lower(),
                rel.get('head_type', '').strip().lower(),
                rel.get('relation', '').strip().lower(),
                rel.get('tail', '').strip().lower(),
                rel.get('tail_type', '').strip().lower()
            )
            for rel in kg
        )

    set1 = extract_core_relations(kg1)
    set2 = extract_core_relations(kg2)

    intersection_size = len(set1.intersection(set2))
    union_size = len(set1.union(set2))

    # Calculate Jaccard similarity (0.0 to 1.0)
    similarity_score = intersection_size / union_size

    return float(similarity_score)

In [3]:
original_folder = "/scratch/lamdo/IRB/runs/22March2026/step4_extracted_kg_checked (main)/"
new_folder = "/scratch/lamdo/IRB/runs/22March2026/step4_extracted_kg_checked/"

files = os.listdir(new_folder)
sims = []
check = []
for file in files:
    original_full_path = os.path.join(original_folder, file)
    new_full_path = os.path.join(new_folder, file)

    with open(original_full_path) as f:
        original_data = json.load(f)

    with open(new_full_path) as f:
        new_data = json.load(f)

    original_fact_kg_mapper = original_data.get("fact_kg_mapper")
    new_fact_kg_mapper = new_data.get("fact_kg_mapper")

    for k in new_fact_kg_mapper:
        temp = calculate_kg_similarity(original_fact_kg_mapper.get(k, []), new_fact_kg_mapper.get(k, []))
        if temp < 1.0: 
            check.append((file, k, original_fact_kg_mapper.get(k, []), new_fact_kg_mapper.get(k, [])))
        sims.append(temp)

In [4]:
import numpy as np
np.mean(sims)

np.float64(0.8098412698412697)

In [5]:
len([item for item in check if len(item[2]) > len(item[3])])

26

In [6]:
len(check)

32

In [7]:
[item for item in check if len(item[2]) < len(item[3])][0]

('Yoram Zague.json',
 '6',
 [{'head': 'Yoram Zague',
   'head_type': 'Person',
   'relation': 'scored his first goal for',
   'tail': 'Paris Saint-Germain (PSG)',
   'tail_type': 'Soccer team',
   'head_coverage': [0],
   'tail_coverage': [0]},
  {'head': 'Yoram Zague',
   'head_type': 'Person',
   'relation': 'scored on his birthday',
   'tail': '15 May 2024',
   'tail_type': 'Date',
   'head_coverage': [0],
   'tail_coverage': [0]},
  {'head': '15 May 2024',
   'head_type': 'Date',
   'relation': "was Yoram Zague's",
   'tail': '18th birthday',
   'tail_type': 'Birthday',
   'head_coverage': [0],
   'tail_coverage': [0]}],
 [{'head': 'Yoram Zague',
   'head_type': 'Person',
   'relation': 'scored his first goal for',
   'tail': 'Paris Saint-Germain (PSG)',
   'tail_type': 'Soccer team',
   'head_coverage': [0],
   'tail_coverage': [0]},
  {'head': '2-1 win over Nice',
   'head_type': 'Match result',
   'relation': 'was against',
   'tail': 'Nice',
   'tail_type': 'Soccer team',
   'h